# ResNet18, pretrained on ImageNet

Model 3 of 5. A ResNet18 with ImageNet weights; the last layer is replaced by Dropout(0.5) -> Linear(512,15)
and all layers are fine-tuned. Recipe as in the original `train.py`: Adam, learning rate 1e-4,
random batches of 32, early stopping with patience 10.

* Input: the whole image, only resized to 512x512. Target: the class id from the `.txt` label file.
* Scene-wise split, identical for all models: 116 / 25 / 25 scenes (1740 / 375 / 375 images).
* One full training run per seed; the test accuracy is reported as mean +- std over the seeds.
* All code lives in the repository folder `src/`; this notebook only calls it.
* Outputs: `outputs/<model>/weights/`, `figures/` and `results/`; saved to GitHub (and Drive in Colab) at the end.

**Colab:** Runtime -> Change runtime type -> **GPU**, then Run all.  
**Own computer:** see the README (one-time setup), then Run all.

## 1. Setup: code and data

In [ ]:
# Works in Colab and on an own computer with an NVIDIA GPU (e.g. the university computer).
import os, sys
REPO_URL = "https://github.com/Ahmed-AlTahan/CNN-LIME.git"

if "google.colab" in sys.modules:          # Colab: get the code from GitHub
    REPO_DIR = "/content/thesis"
    if not os.path.isdir(REPO_DIR):
        !git clone -q {REPO_URL} {REPO_DIR}
else:                                      # own computer: this notebook is in <repository>/notebooks/
    REPO_DIR = os.path.abspath("..") if os.path.basename(os.getcwd()) == "notebooks" else os.getcwd()
!git -C "{REPO_DIR}" pull -q --ff-only
os.chdir(REPO_DIR)
sys.path.insert(0, os.path.join(REPO_DIR, "src"))

import env
env.ensure_dataset()      # downloads the dataset into data/ the first time
env.print_info()

## 2. Settings
The recipe comes from `config.RECIPES`; change it here for an experiment.

In [ ]:
import config, data, engine, report

recipe = config.RECIPES["resnet18"]
# Change settings here if needed, e.g.:
# recipe = config.replace(recipe, seeds=[1, 2, 3, 4, 5])
out_dir = report.output_dir(recipe.model)
print(recipe)
print("outputs ->", engine.rel(out_dir))

## 3. Load data
Every image is decoded once and kept in memory (a few minutes).

In [ ]:
train_ds, val_ds, test_ds = data.load_splits()
val_loader = data.make_eval_loader(val_ds, recipe.batch_size)
test_loader = data.make_eval_loader(test_ds, recipe.batch_size)

## 4. Check the inputs
Augmented training images, exactly as the network sees them.

In [ ]:
report.plot_inputs(train_ds, recipe, out_dir)

## 5. Sanity check
A fresh model must overfit one scene (15 images) to ~100%. If it cannot, the pipeline is broken.

In [ ]:
engine.sanity_check(recipe, train_ds)

## 6. Training
One run per seed. Re-running this cell gives the same results.

In [ ]:
runs = engine.train_all(recipe, train_ds, val_loader, out_dir)
report.plot_curves(runs, out_dir)

## 7. Test set
Every run is evaluated once on the test set. The detailed analysis uses the run with the best **validation** accuracy.

In [ ]:
best = report.test_summary(runs, test_loader, recipe, out_dir)
scene_acc = report.analyse(best, test_ds, out_dir, title="ResNet18, pretrained on ImageNet")

## 8. Save the results
* **GitHub:** figures and result files (no weights) go to `results/resnet18/` in the repository and are pushed to the main branch.
* **Colab only - Google Drive:** a full copy including the weights goes to `MyDrive/thesis_results/resnet18/`. On an own computer the weights stay in `outputs/resnet18/weights/`.

A re-run replaces the files of this model only. Every file name starts with the model name.

**Colab, once:** to push from Colab, add a secret named `GITHUB_TOKEN` (key icon in the left sidebar) holding a GitHub personal access token with *Contents: read and write* for this repository, and allow notebook access. Without it the results are still saved on Drive.

In [ ]:
env.save_results(out_dir)